In [15]:
import os
from pathlib import Path
import pandas as pd

In [16]:
PROJECT_ROOT = Path().resolve().parents[1]
DATA_DIR = PROJECT_ROOT / os.getenv("DATA_DIR")

In [17]:
df = pd.read_csv(DATA_DIR/"raw"/"healthcare-dataset-stroke-data.csv")

In [18]:
num_cols = ["age","avg_glucose_level","bmi"]
cat_cols = ["gender","ever_married","smoking_status","work_type","Residence_type"]

In [19]:
df = df.drop(columns=["id"])
df["bmi"] = df["bmi"].fillna(df["bmi"].median())

In [20]:
#Label encoding for binary variables
df["gender"] = df["gender"].map({"Male":0, "Other": 0, "Female":1})
df["ever_married"] = df["ever_married"].map({"No":0, "Yes":1})
df["Residence_type"] = df["Residence_type"].map({"Rural":0, "Urban":1})

In [21]:
#One hot encoding for categorical variables
df = pd.get_dummies(df,columns=["work_type","smoking_status"], drop_first=True)

In [22]:
df.columns

Index(['gender', 'age', 'hypertension', 'heart_disease', 'ever_married',
       'Residence_type', 'avg_glucose_level', 'bmi', 'stroke',
       'work_type_Never_worked', 'work_type_Private',
       'work_type_Self-employed', 'work_type_children',
       'smoking_status_formerly smoked', 'smoking_status_never smoked',
       'smoking_status_smokes'],
      dtype='str')

In [23]:
#Define X (predictor) and y (Target) variables
X_reduced = df[["age","hypertension","heart_disease","avg_glucose_level",
                "bmi","smoking_status_formerly smoked","smoking_status_smokes"]]

y = df["stroke"]

In [24]:
#Train/Test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_reduced, y, test_size=0.2, random_state=42, stratify=y
)

In [25]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

In [26]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual No Stroke (TN/FP)", "Actual Stroke (FN/TP)"],
    columns=["Predicted No stroke", "Predicted Stroke"]

)

print(cm_df)

print("\nClassification Report: \n", classification_report(y_test, y_pred))

Accuracy: 0.7407045009784736
                          Predicted No stroke  Predicted Stroke
Actual No Stroke (TN/FP)                  717               255
Actual Stroke (FN/TP)                      10                40

Classification Report: 
               precision    recall  f1-score   support

           0       0.99      0.74      0.84       972
           1       0.14      0.80      0.23        50

    accuracy                           0.74      1022
   macro avg       0.56      0.77      0.54      1022
weighted avg       0.94      0.74      0.81      1022



Precision (class 0) = TN / (TN + FN)

In [27]:
X = df.drop(columns=["stroke"])

In [28]:
coeffs = pd.Series(model.coef_[0],index=X_reduced.columns)
coeffs = coeffs.abs().sort_values(ascending=False)
coeffs

age                               1.754632
hypertension                      0.551012
smoking_status_smokes             0.365235
heart_disease                     0.281903
smoking_status_formerly smoked    0.208128
avg_glucose_level                 0.196883
bmi                               0.024843
dtype: float64

In [29]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("Accuracy", accuracy_score(y_test, y_pred_rf))

cm = confusion_matrix(y_test, y_pred_rf)
cm_df = pd.DataFrame(
    cm,
    index=["Actual No Stroke (TN/FP)", "Actual Stroke (FN/TP)"],
    columns=["Predicted No stroke", "Predicted Stroke"]

)
print(cm_df)

print(classification_report(y_test, y_pred_rf))

Accuracy 0.9500978473581213
                          Predicted No stroke  Predicted Stroke
Actual No Stroke (TN/FP)                  971                 1
Actual Stroke (FN/TP)                      50                 0
              precision    recall  f1-score   support

           0       0.95      1.00      0.97       972
           1       0.00      0.00      0.00        50

    accuracy                           0.95      1022
   macro avg       0.48      0.50      0.49      1022
weighted avg       0.90      0.95      0.93      1022



In [ ]:
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

In [ ]:
for col in num_cols:
    print(col)
    print("Minimum ", df_stroke[col].min())
    print("Maximum ", df_stroke[col].max())
    print("Mean ", df_stroke[col].mean())
    print("Median", df_stroke[col].median())

In [ ]:
for col in cat_cols:
    print(df_stroke[col].value_counts(normalize=True))

In [ ]:
df_stroke_yes = df_stroke.loc[df_stroke["stroke"]== 1, num_cols+cat_cols]
df_stroke_yes

In [ ]:
for col in cat_cols:
    print(df_stroke_yes[col].value_counts(normalize=True))
   

In [ ]:
#df_stroke[['age','hypertension','heart_disease','avg_glucose_level','bmi','stroke']].corr(method='pearson')

In [ ]:
#sns.heatmap() is used to visualize the correlation matrix
# Visualize the correlation matrix usign heat map.
correlation_matrix = df_stroke[['age','hypertension','heart_disease','avg_glucose_level','bmi','stroke']].corr(method='pearson') # Compute the correlation matrix

plt.figure(figsize=(5, 6)) # Adjust plot size
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f') # Create the heatmap
plt.title('Correlation Heatmap - Linear (Pearson)') # Add a title
plt.show() # Display the plot


In [ ]:
df_stroke['bmi'] = df_stroke['bmi'].fillna(df_stroke['bmi'].median())
df_stroke_model = df_stroke.drop(columns=['id'])
df_stroke_model1 = df_stroke.drop(columns=['id'])

In [ ]:
from sklearn.preprocessing import OneHotEncoder

oh = OneHotEncoder(sparse_output=False, handle_unknown='ignore',drop='first')
encoded = oh.fit_transform(df_stroke_model[cat_cols])
encoded_df = pd.DataFrame(encoded, columns=oh.get_feature_names_out(cat_cols))

df_stroke_model = pd.concat([df_stroke_model.drop(columns=cat_cols), encoded_df], axis=1)


In [ ]:
# Correlation Analysis : Compute the Pearson correlation coefficient between every numerical column in df_model and the 'stroke' column.
plt.figure(figsize=(10, 6))
correlation = df_stroke_model.corr()['stroke'].sort_values(ascending=False) # Sort the resulting correlation values in descending order
sns.barplot(x=correlation.values, y=correlation.index, palette='viridis', hue=correlation.index, legend=False)
plt.title('Correlation of Features with Stroke')
plt.xlabel('Correlation Coefficient')
plt.show()

In [ ]:
X = df_stroke_model.drop('stroke', axis=1)
y = df_stroke_model['stroke']

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X,y)

feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
}).sort_values(by='Importance', ascending=False)

feature_importance

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=feature_importance.head(5), x='Importance', y='Feature', palette='magma', hue='Feature', legend=False)
plt.title('Top 3 Predictors of Stroke (Feature Importance)')
plt.tight_layout()


In [ ]:
# Heatmap of Risk Age and Glucose
# Create age bins
df_stroke['age_bin'] = pd.cut(df_stroke['age'], bins=[0, 40, 60, 80, 100], labels=['0-40', '40-60', '60-80', '80+'])
# Create glucose bins
df_stroke['glucose_bin'] = pd.cut(df_stroke['avg_glucose_level'], bins=[0, 90, 140, 200, 300], labels=['Low', 'Normal', 'High', 'Very High'])

# Pivot table for heatmap
pivot = df_stroke.pivot_table(index='age_bin', columns='glucose_bin', values='stroke', aggfunc='mean', observed=False)

sns.heatmap(pivot, annot=True, cmap='Reds', fmt='.2%')
plt.title('Stroke Probability: Age vs. Glucose Levels')
plt.show()